# Integrating Tools into an Agent: Practice Exercise

In this exercise, you'll integrate the **Wikipedia search tool** into a LangChain agent. This reinforces the pattern of adding external tools to give agents real-world capabilities.

**What you'll implement:**
- Configure and instantiate a Wikipedia search tool
- Create an agent with the tool integrated

**Estimated time:** 10 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
%pip install -qU langchain-openai
%pip install -U langchain

In [2]:
%pip install -qU langchain-community

In [8]:
%pip install -qU wikipedia

  Preparing metadata (setup.py) ... done


In [4]:
# Setup - run this cell first

import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Load environment variables
from google.colab import userdata

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    print("OpenAI API key found. Ready to proceed!")
else:
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

## Context

You are building a research assistant that can look up factual information from Wikipedia. The agent should be able to answer questions about historical events, scientific concepts, notable people, and other encyclopedic topics.

**Your task:**
- Create a Wikipedia search tool using `WikipediaQueryRun` and `WikipediaAPIWrapper`

**Tool configuration:**
- The `WikipediaAPIWrapper` accepts `top_k_results` (number of results) and `doc_content_chars_max` (max characters per result)
- Use `top_k_results=3` and `doc_content_chars_max=2000` for balanced results

**Expected behavior:**
- The tool should be able to query Wikipedia and return relevant information
- We'll provide the agent setup for you - you just need to create the tool

## Your Task: Create the Wikipedia Search Tool

Implement a function that creates and returns a configured Wikipedia search tool.

In [14]:
def create_wikipedia_tool():
    """
    Create a Wikipedia search tool for querying encyclopedic information.

    The tool should be configured with:
    - top_k_results: 3 (return top 3 Wikipedia articles)
    - doc_content_chars_max: 2000 (limit content length per result)

    Returns:
        WikipediaQueryRun: A configured Wikipedia search tool instance

    Hint: WikipediaQueryRun takes an api_wrapper parameter
    """
    # TODO: Create and return the Wikipedia search tool
    wiki_wrapper = WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=2000)
    wiki_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)

    return wiki_tool

## Agent Setup

Below is the agent setup code - you don't need to modify this. It will use your Wikipedia tool.

In [22]:
# Agent setup - run this after creating your tool

def create_research_agent(wiki_tool):
    """
    Create a LangChain agent with Wikipedia search capability.

    Args:
        wiki_tool: A WikipediaQueryRun tool instance

    Returns:
        Agent: A configured LangChain agent
    """
    model = ChatOpenAI(model="gpt-4o", temperature=0.1, api_key=OPEN_AI_API_KEY)
    agent = create_agent(model=model, tools=[wiki_tool])
    return agent

## Test Your Implementation

Run the cells below to test your Wikipedia tool with the agent.

In [16]:
# Create the Wikipedia tool
wiki_tool = create_wikipedia_tool()
print(f"Tool created: {wiki_tool.name}")
print(f"Description: {wiki_tool.description[:100]}...")

Tool created: wikipedia
Description: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, place...


In [23]:
# Create the agent
research_agent = create_research_agent(wiki_tool)
print("Research agent created successfully!")

Research agent created successfully!


In [24]:
# Test with a factual query
query = "Who invented the telephone and when?"
print(f"Query: {query}\n")
print("=" * 60)

# Invoke the agent and capture the result
result = research_agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

# Extract and display tool calls
print("\n🔧 TOOL CALLS MADE:")
print("-" * 60)
for msg in result['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tool_call in msg.tool_calls:
            print(f"Tool: {tool_call['name']}")
            print(f"Query: {tool_call['args']['query']}")
            print()

# Display tool responses
print("📚 TOOL RESPONSES:")
print("-" * 60)
for msg in result['messages']:
    if hasattr(msg, 'name') and msg.name == 'wikipedia':
        # Truncate long responses for display
        content_preview = msg.content[:500] + "..." if len(msg.content) > 500 else msg.content
        print(content_preview)
        print()

# Display final answer
print("💬 FINAL ANSWER:")
print("-" * 60)
print(f"{result['messages'][-1].content}")
print("=" * 60)

Query: Who invented the telephone and when?


🔧 TOOL CALLS MADE:
------------------------------------------------------------
Tool: wikipedia
Query: invention of the telephone

📚 TOOL RESPONSES:
------------------------------------------------------------
Page: Invention of the telephone
Summary: The invention of the telephone was the culmination of work done by many different people, and led to an array of lawsuits relating to the conflicting patent claims made by several individuals and numerous companies. Notable people included in this process were Antonio Meucci, Philipp Reis, Elisha Gray and Alexander Graham Bell.



Page: History of the telephone
Summary: This history of the telephone chronicles the development of the electrical telephone,...

💬 FINAL ANSWER:
------------------------------------------------------------
The telephone was invented by Alexander Graham Bell, who was granted the first telephone patent in 1876.
